# LLM 15: Hands-On — Claude + OpenAI + Local

Build the minimum viable multi-provider client and benchmark it.

1. Unified `LLMResponse` dataclass
2. `ClaudeClient`, `OpenAIClient`, `OllamaClient` adapters
3. Run the same prompt through all three; compare cost/latency/quality
4. Fallback decorator
5. Exercises: streaming unified interface, tool-use unification

Deps: `pip install anthropic openai`. For local: `pip install ollama` or `vLLM`, then pull `llama3`.

## 1. The Unified Response Type

In [ ]:
from dataclasses import dataclass
from typing import Any, Optional
import time

PRICES = {
    'claude-sonnet-4-6': (3.00, 15.00),
    'claude-haiku-4-5-20251001': (1.00, 5.00),
    'gpt-4o-mini': (0.15, 0.60),
    'gpt-4o': (2.50, 10.00),
    'llama3': (0.00, 0.00),   # local
}

def compute_cost(model, in_tok, out_tok):
    if model not in PRICES: return 0.0
    i, o = PRICES[model]
    return (in_tok * i + out_tok * o) / 1_000_000

@dataclass
class LLMResponse:
    text: str
    input_tokens: int
    output_tokens: int
    cached_tokens: int
    cost_usd: float
    latency_ms: float
    provider: str
    model: str
    raw: Any = None

## 2. Provider Adapters

In [ ]:
class ClaudeClient:
    def __init__(self, model='claude-sonnet-4-6'):
        from anthropic import Anthropic
        self.c = Anthropic()
        self.model = model
    def generate(self, system, user, *, temperature=0.2, max_tokens=1024) -> LLMResponse:
        t0 = time.perf_counter()
        r = self.c.messages.create(
            model=self.model, system=system, max_tokens=max_tokens,
            temperature=temperature,
            messages=[{'role':'user','content':user}])
        return LLMResponse(
            text=r.content[0].text,
            input_tokens=r.usage.input_tokens,
            output_tokens=r.usage.output_tokens,
            cached_tokens=getattr(r.usage, 'cache_read_input_tokens', 0) or 0,
            cost_usd=compute_cost(self.model, r.usage.input_tokens, r.usage.output_tokens),
            latency_ms=(time.perf_counter()-t0)*1000,
            provider='anthropic', model=self.model, raw=r)

class OpenAIClient:
    def __init__(self, model='gpt-4o-mini', base_url=None):
        from openai import OpenAI
        self.c = OpenAI(base_url=base_url) if base_url else OpenAI()
        self.model = model
    def generate(self, system, user, *, temperature=0.2, max_tokens=1024) -> LLMResponse:
        t0 = time.perf_counter()
        r = self.c.chat.completions.create(
            model=self.model, max_tokens=max_tokens, temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        u = r.usage
        return LLMResponse(
            text=r.choices[0].message.content,
            input_tokens=u.prompt_tokens,
            output_tokens=u.completion_tokens,
            cached_tokens=getattr(u, 'prompt_tokens_details', None).cached_tokens if getattr(u, 'prompt_tokens_details', None) else 0,
            cost_usd=compute_cost(self.model, u.prompt_tokens, u.completion_tokens),
            latency_ms=(time.perf_counter()-t0)*1000,
            provider='openai', model=self.model, raw=r)

class OllamaClient(OpenAIClient):
    def __init__(self, model='llama3'):
        super().__init__(model=model, base_url='http://localhost:11434/v1')
        self.c.api_key = 'ollama'  # Ollama ignores auth
        self.provider_name = 'ollama'

## 3. Run the Same Prompt Through All Three

In [ ]:
import os

clients = {}
if os.getenv('ANTHROPIC_API_KEY'):
    clients['claude-sonnet-4-6'] = ClaudeClient('claude-sonnet-4-6')
if os.getenv('OPENAI_API_KEY'):
    clients['gpt-4o-mini']  = OpenAIClient('gpt-4o-mini')
# Uncomment if Ollama is running locally:
# clients['ollama-llama3']  = OllamaClient('llama3')

system = 'You are a helpful technical writer. Keep answers under 50 words.'
user = 'Explain what a Kubernetes pod is.'

print(f"{'model':<30} {'ms':>6} {'in tok':>7} {'out tok':>7} {'cost $':>10}  text")
for name, client in clients.items():
    try:
        r = client.generate(system, user)
        print(f'{name:<30} {r.latency_ms:>6.0f} {r.input_tokens:>7} {r.output_tokens:>7} ${r.cost_usd:>9.5f}  {r.text[:60]}...')
    except Exception as e:
        print(f'{name:<30} error: {e}')

## 4. Fallback Decorator

In [ ]:
class FallbackClient:
    def __init__(self, *clients):
        self.clients = clients
    def generate(self, *a, **kw):
        last = None
        for c in self.clients:
            try:
                return c.generate(*a, **kw)
            except Exception as e:
                last = e
        raise last

if len(clients) >= 2:
    primary, secondary = list(clients.values())[:2]
    resilient = FallbackClient(primary, secondary)
    r = resilient.generate(system, user)
    print(f'resilient response via {r.provider}: {r.text[:80]}...')
else:
    print('Need at least 2 providers configured to demo fallback.')

## 5. Quality Comparison via LLM-as-Judge

In [ ]:
eval_set = [
    'Explain continuous integration in two sentences.',
    'Why use HTTPS instead of HTTP?',
    'What is the CAP theorem?',
    'In which year was Linux released?',
    'What does idempotent mean in HTTP?',
]

if 'claude-sonnet-4-6' in clients:
    judge = ClaudeClient('claude-sonnet-4-6')
    per_client_total = {name: {'cost': 0, 'latency': 0, 'wins': 0} for name in clients}

    for q in eval_set:
        answers = {}
        for name, c in clients.items():
            try:
                r = c.generate('Be accurate and concise.', q)
                answers[name] = r
                per_client_total[name]['cost']    += r.cost_usd
                per_client_total[name]['latency'] += r.latency_ms
            except Exception as e:
                print(f'{name} failed: {e}')

        if len(answers) >= 2:
            list_block = '\n\n'.join(f'[{n}]\n{a.text}' for n, a in answers.items())
            verdict = judge.generate(
                'Judge which answer is most accurate and concise. Return the model name only.',
                f'Question: {q}\n\n{list_block}')
            winner = verdict.text.strip()
            for name in answers:
                if name in winner:
                    per_client_total[name]['wins'] += 1
                    break

    print(f"\n{'model':<30} {'wins':>5} {'total cost':>12} {'avg ms':>10}")
    for name, d in per_client_total.items():
        print(f'{name:<30} {d["wins"]:>5} ${d["cost"]:>11.5f} {d["latency"]/len(eval_set):>9.0f}')

## 6. Exercise: Unified Streaming Interface

Extend each client with a `stream(system, user, **kw) -> Iterator[str]` method.
- Claude: use `messages.stream()`, iterate over `text_stream`.
- OpenAI: `stream=True`, iterate over `choices[0].delta.content`.
- Ollama: same as OpenAI.

Ensure each yields just text chunks. Consumer code shouldn't need to know the provider.

## 7. Exercise: Unified Tool Use

Define a canonical `ToolDef = {name, description, parameters}` (JSON Schema). Write a `translate_tool(tool, provider)` function that turns it into Anthropic / OpenAI / Gemini tool format.

Make `generate(..., tools=[...])` return a canonical `ToolCall` object regardless of provider. This is the foundation of a real agent runtime.

## 8. Exercise: Provider Routing by Difficulty

Write a `RoutedClient` that:
1. Classifies incoming prompts with a cheap Haiku/4o-mini call.
2. Routes "simple" prompts to Haiku / 4o-mini.
3. Routes "complex" prompts to Sonnet / 4o.

Compare total cost vs always-Sonnet on 100 mixed prompts.

## 9. Module 05 Wrap

You now have:

- A mental model of transformers, tokenization, embeddings
- Pretraining objectives and decoding strategies you can reason about
- Prompt-engineering basics and structured-output patterns
- Context window awareness + streaming UX
- An evaluation + cost + hallucination mitigation stack
- A multi-provider client abstraction that scales with you

**Next module:** `06-prompt-engineering/` — the deep dive that session 07 only previewed. You'll build prompt libraries, prompt optimization with DSPy, and CI-grade prompt regression tests.